In [3]:
# ============================================================
# EXTREME LEARNING MACHINE (ELM)
# FULL IMPLEMENTATION + HYPERPARAMETER TUNING
# + STABILITY ANALYSIS + COMPUTATION TIME
# ============================================================

import numpy as np
import pandas as pd
import time
import pickle
import warnings
from itertools import product
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score,
    roc_auc_score, roc_curve
)

warnings.filterwarnings("ignore")


# ============================================================
# LOAD DATA
# ============================================================

def load_cleaned_data(filename='data_cleaned.pkl'):
    print("Loading cleaned data...")
    with open(filename, 'rb') as f:
        data = pickle.load(f)
    return data


# ============================================================
# ELM MODEL
# ============================================================

def elm_model(X_train, y_train, X_test,
              n_hidden=50,
              activation='sigmoid',
              reg_lambda=1e-8,
              random_seed=42):

    np.random.seed(random_seed)

    n_features = X_train.shape[1]

    W = np.random.normal(0, 1, (n_features, n_hidden))
    b = np.random.normal(0, 1, n_hidden)

    H = np.dot(X_train, W) + b

    if activation == 'sigmoid':
        H = 1 / (1 + np.exp(-np.clip(H, -250, 250)))
    elif activation == 'tanh':
        H = np.tanh(H)
    elif activation == 'relu':
        H = np.maximum(0, H)

    I = np.identity(H.shape[1])
    beta = np.linalg.pinv(H.T @ H + reg_lambda * I) @ H.T @ y_train

    H_test = np.dot(X_test, W) + b

    if activation == 'sigmoid':
        H_test = 1 / (1 + np.exp(-np.clip(H_test, -250, 250)))
    elif activation == 'tanh':
        H_test = np.tanh(H_test)
    elif activation == 'relu':
        H_test = np.maximum(0, H_test)

    y_pred = H_test @ beta
    y_proba = 1 / (1 + np.exp(-np.clip(y_pred, -250, 250)))

    return y_proba


# ============================================================
# THRESHOLD OPTIMIZATION
# ============================================================

def find_optimal_threshold(y_true, y_proba):
    fpr, tpr, thresholds = roc_curve(y_true, y_proba)
    J = tpr - fpr
    return thresholds[np.argmax(J)]


# ============================================================
# CROSS VALIDATION
# ============================================================

def run_cross_validation(X, y, preprocessor, elm_params, n_splits=10):

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    results = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):

        start_time = time.time()

        X_train_raw = X.iloc[train_idx]
        X_test_raw = X.iloc[test_idx]
        y_train = y[train_idx]
        y_test = y[test_idx]

        X_train = preprocessor.fit_transform(X_train_raw)
        X_test = preprocessor.transform(X_test_raw)

        y_proba = elm_model(
            X_train, y_train, X_test,
            n_hidden=elm_params['n_hidden'],
            activation=elm_params['activation'],
            reg_lambda=elm_params['reg_lambda'],
            random_seed=elm_params['random_seed']
        )

        threshold = find_optimal_threshold(y_test, y_proba)
        y_pred = (y_proba >= threshold).astype(int)

        results.append({
            "Fold": fold,
            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred, zero_division=0),
            "Recall": recall_score(y_test, y_pred, zero_division=0),
            "F1 Score": f1_score(y_test, y_pred, zero_division=0),
            "AUC": roc_auc_score(y_test, y_proba),
            "Threshold": threshold,
            "Training Time (s)": time.time() - start_time
        })

    return pd.DataFrame(results)


# ============================================================
# GRID SEARCH
# ============================================================

def grid_search_elm(X, y, preprocessor, param_grid, n_splits=10):

    combinations = list(product(
        param_grid['n_hidden'],
        param_grid['activation'],
        param_grid['reg_lambda'],
        param_grid['random_seed']
    ))

    all_results = []

    print("\nGRID SEARCH STARTED")
    print("=" * 60)

    for i, (n_hidden, activation, reg_lambda, seed) in enumerate(combinations, 1):

        print(f"[{i}/{len(combinations)}] Hidden={n_hidden}, Act={activation}, Lambda={reg_lambda}")

        elm_params = {
            "n_hidden": int(n_hidden),
            "activation": activation,
            "reg_lambda": float(reg_lambda),
            "random_seed": int(seed)
        }

        cv_df = run_cross_validation(X, y, preprocessor, elm_params, n_splits)

        all_results.append({
            "Hidden Units": elm_params["n_hidden"],
            "Activation": elm_params["activation"],
            "Reg Lambda": elm_params["reg_lambda"],
            "Accuracy": cv_df["Accuracy"].mean(),
            "Precision": cv_df["Precision"].mean(),
            "Recall": cv_df["Recall"].mean(),
            "F1 Score": cv_df["F1 Score"].mean(),
            "AUC": cv_df["AUC"].mean()
        })

    results_df = pd.DataFrame(all_results)
    results_df = results_df.sort_values(
        by=["F1 Score", "Recall"],
        ascending=False
    ).reset_index(drop=True)

    return results_df


# ============================================================
# STABILITY CALCULATION
# ============================================================

def calculate_stability(cv_df):

    metrics = ["Accuracy", "Precision", "Recall", "F1 Score", "AUC"]

    summary = []

    for metric in metrics:
        mean_val = cv_df[metric].mean()
        std_val = cv_df[metric].std()

        cv_percent = (std_val / mean_val) * 100 if mean_val != 0 else 0

        if cv_percent < 5:
            status = "SANGAT STABIL"
        elif cv_percent < 10:
            status = "STABIL"
        else:
            status = "KURANG STABIL"

        summary.append({
            "Metric": metric,
            "Mean": mean_val,
            "Std Dev": std_val,
            "CV (%)": cv_percent,
            "Stability": status
        })

    summary_df = pd.DataFrame(summary)

    time_mean = cv_df["Training Time (s)"].mean()
    time_total = cv_df["Training Time (s)"].sum()

    return summary_df, time_mean, time_total


# ============================================================
# MAIN
# ============================================================

def main():

    print("="*70)
    print("EXTREME LEARNING MACHINE - HYPERPARAMETER TUNING + STABILITY")
    print("="*70)

    data_loaded = load_cleaned_data("data_cleaned.pkl")

    data_cleaned = data_loaded['data_cleaned']
    preprocessor = data_loaded['preprocessor']

    X = data_cleaned.drop(columns=["diagnosis_lanjutan"])
    y = data_cleaned["diagnosis_lanjutan"].values

    param_grid = {
        "n_hidden": [50, 100, 300, 500],
        "activation": ["sigmoid", "tanh", "relu"],
        "reg_lambda": [1e-8, 1e-4, 1e-2, 1],
        "random_seed": [9011]
    }

    tuning_results = grid_search_elm(X, y, preprocessor, param_grid)
    tuning_results.to_csv("elm_grid_search_results.csv", index=False)

    best_config = tuning_results.iloc[0]

    elm_params_best = {
        "n_hidden": int(best_config["Hidden Units"]),
        "activation": best_config["Activation"],
        "reg_lambda": float(best_config["Reg Lambda"]),
        "random_seed": 9011
    }

    print("\nBEST CONFIGURATION:")
    print(elm_params_best)

    cv_best = run_cross_validation(X, y, preprocessor, elm_params_best)

    print("\nDETAIL 10-FOLD CROSS VALIDATION")
    print(cv_best.to_string(index=False))

    summary_df, time_mean, time_total = calculate_stability(cv_best)

    performance_mean = cv_best[[
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score",
        "AUC"
    ]].mean()

    stability_percent = (
        cv_best[["Accuracy","Precision","Recall","F1 Score","AUC"]]
        .std() /
        cv_best[["Accuracy","Precision","Recall","F1 Score","AUC"]]
        .mean()
    ).mean() * 100

    model_summary = pd.DataFrame([{
        "Model": "ELM",
        "Accuracy": performance_mean["Accuracy"],
        "Precision": performance_mean["Precision"],
        "Recall": performance_mean["Recall"],
        "F1 Score": performance_mean["F1 Score"],
        "AUC": performance_mean["AUC"],
        "Stability (%)": stability_percent,
        "Avg Time (s)": time_mean,
        "Total Time (s)": time_total
    }])

    model_summary.to_csv("elm_model_summary.csv", index=False)

    # tambahkan waktu komputasi ke tabel
    time_row = pd.DataFrame([{
        "Metric": "Training Time",
        "Mean": time_mean,
        "Std Dev": cv_best["Training Time (s)"].std(),
        "CV (%)": 0,
        "Stability": "-"
    }])

    summary_df = pd.concat([summary_df, time_row], ignore_index=True)

    print("\nRINGKASAN PERFORMA + STABILITAS")
    print(summary_df.to_string(index=False))

    print("\nWAKTU KOMPUTASI:")
    print(f"Rata-rata per fold : {time_mean:.4f} detik")
    print(f"Total waktu        : {time_total:.4f} detik")

    summary_df.to_csv("elm_stability_summary.csv", index=False)

    print("\n✅ SELESAI")
    print("File disimpan:")
    print("- elm_grid_search_results.csv")
    print("- elm_stability_summary.csv")


if __name__ == "__main__":
    main()

EXTREME LEARNING MACHINE - HYPERPARAMETER TUNING + STABILITY
Loading cleaned data...

GRID SEARCH STARTED
[1/48] Hidden=50, Act=sigmoid, Lambda=1e-08
[2/48] Hidden=50, Act=sigmoid, Lambda=0.0001
[3/48] Hidden=50, Act=sigmoid, Lambda=0.01
[4/48] Hidden=50, Act=sigmoid, Lambda=1
[5/48] Hidden=50, Act=tanh, Lambda=1e-08
[6/48] Hidden=50, Act=tanh, Lambda=0.0001
[7/48] Hidden=50, Act=tanh, Lambda=0.01
[8/48] Hidden=50, Act=tanh, Lambda=1
[9/48] Hidden=50, Act=relu, Lambda=1e-08
[10/48] Hidden=50, Act=relu, Lambda=0.0001
[11/48] Hidden=50, Act=relu, Lambda=0.01
[12/48] Hidden=50, Act=relu, Lambda=1
[13/48] Hidden=100, Act=sigmoid, Lambda=1e-08
[14/48] Hidden=100, Act=sigmoid, Lambda=0.0001
[15/48] Hidden=100, Act=sigmoid, Lambda=0.01
[16/48] Hidden=100, Act=sigmoid, Lambda=1
[17/48] Hidden=100, Act=tanh, Lambda=1e-08
[18/48] Hidden=100, Act=tanh, Lambda=0.0001
[19/48] Hidden=100, Act=tanh, Lambda=0.01
[20/48] Hidden=100, Act=tanh, Lambda=1
[21/48] Hidden=100, Act=relu, Lambda=1e-08
[22/48] 